# Moving Correlation Significance Tests

This notebook demonstrates the `moving_corr_sig` package: three complementary
significance tests for **moving (windowed) correlation** analyses, each
addressing a different question about a pair of time series `x`, `y`:

| Test | Question | Null conditions on |
|---|---|---|
| `std_test` | Is the running-correlation trace *more* (or *less*) variable through time than sampling noise alone predicts? (Gershunov et al. 2001) | Overall (background) correlation `c` |
| `peak_test` | Is the single highest windowed correlation higher than chance predicts? | `c = 0` (existence) **or** `c = observed` (elevation above baseline) |
| `range_test` | Is the swing between the highest and lowest windowed correlation bigger than chance predicts? | Overall (background) correlation `c` |

Each test can be run against a **white-noise** null (independent Gaussian
samples, as in the original Gershunov et al. 2001 paper) or a **red-noise**
null (AR(1) surrogates matched to the lag-1 autocorrelation of your actual
data) -- important whenever your real series have genuine persistence, which
white-noise nulls will not account for.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from moving_corr_sig import (
    moving_correlation, MovingCorrelationTest,
    std_test, peak_test, range_test,
)
from moving_corr_sig.core import white_noise_pair, red_noise_pair, ar1_fit

plt.rcParams['figure.figsize'] = (9, 3.5)
rng = np.random.default_rng(0)

# Note: this assumes the moving_corr_sig/ folder is in the same directory as
# this notebook (e.g. both extracted from moving_corr_sig.zip together). If
# you get a ModuleNotFoundError here, either move this notebook next to
# moving_corr_sig/, or add its parent folder to sys.path manually:
#   import sys; sys.path.insert(0, r"/path/to/folder/containing/moving_corr_sig")


## 1. Validation against Gershunov et al. (2001), Table 1

The paper's worked example (also reproduced in the `gsbtest.m` docstring
this package is descended from): for two white-noise series with an overall
correlation of `r = 0.30` and an `11`-year window, the bootstrapped 95th
percentile of the standard deviation of the running correlation should be
approximately **0.36**.


In [ ]:
stds = []
for _ in range(3000):
    x, y = white_noise_pair(n=100, target_corr=0.30, rng=rng)
    r = moving_correlation(x, y, window=11)
    stds.append(np.nanstd(r))
stds = np.array(stds)

print(f"95th percentile of null std distribution: {np.quantile(stds, 0.95):.3f}  (paper: ~0.36)")
print(f"99th percentile of null std distribution: {np.quantile(stds, 0.99):.3f}")

plt.hist(stds, bins=40, color='steelblue', alpha=0.8)
plt.axvline(np.quantile(stds, 0.95), color='crimson', ls='--', label='95th pct')
plt.xlabel('std(moving correlation)'); plt.ylabel('count')
plt.title('Null distribution: std of moving correlation, white noise, c=0.30, window=11')
plt.legend(); plt.tight_layout(); plt.show()


## 2. Three synthetic scenarios

We construct three idealized cases where we *know* the ground truth, and
check that the three tests recover the right answer:

- **Scenario A -- No true relationship.** Two independent red-noise series.
  None of the three tests should flag anything real.
- **Scenario B -- Stable relationship.** A constant true correlation the
  whole way through (red noise). `std_test` and `range_test` should *not*
  flag excess variability -- any wobble in the trace is just sampling noise
  around a genuinely constant relationship. `peak_test` (observed-conditioned)
  should not flag an exceptional peak either.
- **Scenario C -- Real regime change.** Correlation is strong in the first
  half of the record and negligible in the second half. All three tests
  should detect this.

Each scenario uses `MovingCorrelationTest`, which computes the moving
correlation once and lets you run any/all of the three tests against it.


In [ ]:
def run_scenario(name, x, y, window, noise, seed):
    mct = MovingCorrelationTest(x, y, window=window, noise=noise, rng=np.random.default_rng(seed))
    print(f"=== {name} ===")
    print(f"overall correlation: {mct.overall_corr:.3f}"
          + (f"   phi_x={mct.phi_x:.2f}, phi_y={mct.phi_y:.2f}" if noise == 'red' else ""))
    results = mct.run_all(iters=1000)
    for test_name, res in results.items():
        tail = f"p_lower={res.p_value_lower:.3f}, " if res.p_value_lower is not None else ""
        print(f"  {test_name:12s} observed={res.observed:.3f}  {tail}p_upper={res.p_value:.3f}")
    print()

    fig, ax = plt.subplots()
    t = np.arange(len(mct.r)) + window - 1
    ax.plot(t, mct.r, color='k')
    ax.axhline(mct.overall_corr, color='gray', ls=':', label='overall correlation')
    ax.set_title(f"{name}: moving correlation (window={window})")
    ax.set_ylim(-1.05, 1.05)
    ax.legend()
    plt.tight_layout(); plt.show()
    return mct, results


In [ ]:
# Scenario A: no true relationship, red noise
xA, yA = red_noise_pair(n=150, target_corr=0.0, phi_x=0.6, phi_y=0.6, rng=rng)
mctA, resA = run_scenario("Scenario A: no true relationship (red noise)", xA, yA, window=15, noise='red', seed=1)


In [ ]:
# Scenario B: constant, stable true relationship, red noise
xB, yB = red_noise_pair(n=150, target_corr=0.6, phi_x=0.6, phi_y=0.6, rng=rng)
mctB, resB = run_scenario("Scenario B: constant true correlation c=0.6 (red noise)", xB, yB, window=15, noise='red', seed=2)


In [ ]:
# Scenario C: real regime change -- strong relationship for the first half, weak for the second
x1, y1 = white_noise_pair(n=75, target_corr=0.85, rng=rng)
x2, y2 = white_noise_pair(n=75, target_corr=0.05, rng=rng)
xC, yC = np.concatenate([x1, x2]), np.concatenate([y1, y2])
mctC, resC = run_scenario("Scenario C: regime change 0.85 -> 0.05 (white noise)", xC, yC, window=15, noise='white', seed=3)


## 3. Peak test: `condition='zero'` vs `condition='observed'`

This illustrates the distinction discussed alongside this package: testing a
peak against a **zero-correlation** null answers "did any relationship exist
at all," while testing against the **observed overall correlation** answers
"is this peak notably elevated above the established background
relationship." Once you already know two things are related overall (as in
Scenario B), the zero-correlation version is much easier to "pass" -- it
isn't really testing anything about the *peak* being special, just that a
real relationship exists somewhere, which you already knew.


In [ ]:
for cond in ("zero", "observed"):
    res = mctB.peak_test(iters=1000, condition=cond)
    print(f"Scenario B peak test, condition='{cond}':")
    print(f"  target_corr={res.target_corr:.3f}  observed max r={res.observed:.3f}  p={res.p_value:.4f}")
print()
print("Notice how much easier the zero-correlation null is to clear once a real background")
print("relationship (c=0.6) already exists -- it mostly just re-detects that relationship,")
print("not anything special about the peak window itself.")


## 4. White noise vs red noise nulls on the same data

If your real series have genuine autocorrelation, a white-noise null will
understate how much a moving correlation trace can wobble by chance alone --
autocorrelated ("red") surrogate series produce systematically *more*
variable running-correlation traces than white noise at the same target
correlation, because neighboring windows overlap in genuinely persistent
signal, not just in shared samples. Using the *wrong* (white) null on
autocorrelated data therefore risks **false positives**: real sampling
wobble around a stable, autocorrelated relationship can look artificially
"significant" if it's compared against a null that doesn't know about the
persistence.

We illustrate this directly with Scenario B's data (phi ~ 0.5, a genuinely
constant true correlation, so nothing *should* be flagged as significant
variability).


In [ ]:
phi_x = ar1_fit(xB)
phi_y = ar1_fit(yB)
print(f"Fitted AR(1) coefficients for Scenario B data: phi_x={phi_x:.3f}, phi_y={phi_y:.3f}")

res_white = std_test(xB, yB, window=15, iters=1500, noise='white', rng=np.random.default_rng(10))
res_red   = std_test(xB, yB, window=15, iters=1500, noise='red',   rng=np.random.default_rng(10))

print()
print("--- Using a WHITE-NOISE null on genuinely autocorrelated data ---")
print(res_white.summary())
print()
print("--- Using the correctly-specified RED-NOISE null ---")
print(res_red.summary())
print()
print("The white-noise null's tighter null distribution can make ordinary sampling")
print("wobble around a stable, autocorrelated relationship look borderline 'significant'")
print("-- the red-noise null (correctly matched to the data's own persistence) shows")
print("this same observed value is entirely unremarkable.")


## 5. Peak-test significance thresholds for plotting

`peak_test`'s null distribution is built from the *maximum* correlation seen
anywhere in each surrogate trace, so a single threshold value (e.g. its 95th
percentile) already accounts for having searched the whole record, and
applies uniformly as a flat horizontal line across the entire observed
trace. This is exactly what you want when the observed moving correlation
mostly sits below the null model but has one or a few genuine excursions
above it -- you can draw the threshold(s) directly alongside the trace and
see at a glance which excursions are real.

We build a synthetic example on purpose: a quiet, unrelated relationship for
most of the record, with one genuine strongly-correlated episode in the
middle.


In [ ]:
x1, y1 = white_noise_pair(n=60, target_corr=0.05, rng=rng)
x2, y2 = white_noise_pair(n=30, target_corr=0.90, rng=rng)   # the genuine episode
x3, y3 = white_noise_pair(n=60, target_corr=0.05, rng=rng)
xD, yD = np.concatenate([x1, x2, x3]), np.concatenate([y1, y2, y3])

mctD = MovingCorrelationTest(xD, yD, window=15, noise='white', rng=np.random.default_rng(20))

# levels= controls which quantiles land in the returned TestResult's .quantiles dict
peak_result = mctD.peak_test(iters=1500, levels=(0.90, 0.95, 0.99))
print(peak_result.summary())
print()

# .threshold() computes ANY quantile on demand, not just the ones precomputed above
print(f"97.5th percentile threshold (not precomputed above): {peak_result.threshold(0.975):.3f}")


In [ ]:
# Ready-made plot: trace + threshold line(s) + highlighted exceedance points
ax, _ = mctD.plot_peak_test(iters=1500, levels=(0.90, 0.95))
ax.set_title("Peak test: quiet relationship with one genuine excursion")
plt.tight_layout(); plt.show()


## 6. Quick reference: running all three tests on your own data

```python
from moving_corr_sig import MovingCorrelationTest

mct = MovingCorrelationTest(seriesA, seriesB, window=15, noise="red")  # or noise="white"

std_result   = mct.std_test(iters=2000)
peak_result  = mct.peak_test(iters=2000, condition="observed")  # or condition="zero"
range_result = mct.range_test(iters=2000)

print(std_result.summary())
print(peak_result.summary())
print(range_result.summary())
```

`noise="red"` fits an AR(1) model to each of your two series automatically
(via `ar1_fit`, which uses statsmodels' `AutoReg`) and builds surrogate pairs
that share both that autocorrelation structure and the chosen target overall
correlation -- so the red-noise null accounts for genuine persistence in your
data the way the original `runningCorrSig.m` / ARFIT pipeline did, but now
integrated with the Gershunov-style variability and range tests as well, not
just the peak test.
